# AAPL next-day price prediction with an LSTM, evaluated honestly

**The question this notebook answers:** does an LSTM predict Apple's next-day close any better than the
free baseline *"tomorrow = today"*? Daily stock prices are famously close to a random walk, so that
baseline is very hard to beat, and any model has to be measured against it.

What is done differently from a typical LSTM-on-prices tutorial:

| Pitfall | What this notebook does |
|---|---|
| Raw `close` contains the 2020-08-31 4:1 split as a fake ~75% crash | Uses split- and dividend-adjusted `adjClose`, and refuses data with unadjusted jumps |
| Modelling price *levels*, which trend out of the scaler's range | Predicts next-day **log returns**, then converts back to dollars |
| Scaler fit on all data; test set used to monitor training | Scaler and weights only ever see the past; separate validation slice for early stopping |
| One split, one unseeded run | **Walk-forward** evaluation (5 folds), several seeds, one seed ensemble |
| No baseline; RMSE mixed scaled and dollar values | RMSE/MAE/MAPE/direction in **dollars** vs persistence, moving averages and ARIMA, with bootstrap confidence intervals |
| One 30-day line with no uncertainty, never checked | A simulated **fan** of paths, then a **backtest** against the closes that actually followed |

> **Educational project, not financial advice.** Nothing here is a trading signal.

**Running it:** set `TIINGO_API_KEY` (free key: <https://www.tiingo.com/account/api/token>) as an environment
variable or a Colab secret. The first run downloads the data into `data/`; later runs use that file.
Set `STOCK_LSTM_QUICK=1` for a fast smoke run (tiny model, 2 folds); the full run trains 15 small networks.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tapankhaladkar/Stock-Prediction-using-LSTM-.git"


def find_repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "stock_lstm").is_dir():
            return p
    return None


ROOT = find_repo_root()
if ROOT is None:  # e.g. a fresh Colab runtime: fetch the repo (the helpers live in stock_lstm/)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "Stock-Prediction-using-LSTM-"], check=True)
    ROOT = Path("Stock-Prediction-using-LSTM-").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from stock_lstm.data import adjustment_events, cache_path, load_prices
from stock_lstm.features import log_returns
from stock_lstm.forecast import make_forecast, residuals_from_predictions, score_forecast
from stock_lstm.model import ModelConfig, fit_model
from stock_lstm.plots import plot_adjusted_vs_raw, plot_forecast, plot_ratio_ci, plot_walk_forward
from stock_lstm.walkforward import WalkForwardConfig, run_walk_forward

# ---- Configuration -------------------------------------------------------------------------
TICKER = "AAPL"
START = "2015-01-01"
ORIGIN = "2025-07-18"   # last day the models may see; the forecast starts the next trading day
END = "2025-09-30"      # data after ORIGIN is held out and used only to score the forecast
HORIZON = 30            # trading days to forecast

QUICK = os.environ.get("STOCK_LSTM_QUICK") == "1"
if QUICK:
    MODEL_CFG = ModelConfig(window=20, units=(8, 8), dropout=0.0, epochs=4, patience=2)
    WF_CFG = WalkForwardConfig(n_folds=2, test_size=60, val_size=60, n_seeds=1)
else:
    MODEL_CFG = ModelConfig(window=60, units=(32, 32), dropout=0.1, epochs=50, patience=6)
    WF_CFG = WalkForwardConfig(n_folds=5, test_size=126, val_size=126, n_seeds=3)
print("QUICK smoke run" if QUICK else "full run", "|", MODEL_CFG, "|", WF_CFG, sep="\n")

## 1. Data: adjusted prices over a fixed date range

`load_prices` uses Tiingo's `adjClose` (continuous across splits and dividends) over an explicit `[START, END]` range, so
every run sees the same history, unlike a rolling "last 5 years" download. It raises if any single-day move looks like an
unadjusted split. Data after `ORIGIN` is set aside so the forecast in section 4 can be scored honestly.

In [ ]:
prices_all = load_prices(TICKER, START, END)

origin = prices_all.loc[:ORIGIN].index[-1]
history = prices_all.loc[:origin]                       # everything the models may use
future = prices_all.loc[prices_all.index > origin]      # realised closes, used only for the backtest

print(f"{len(history)} trading days for modelling: {history.index[0]:%Y-%m-%d} to {origin:%Y-%m-%d}")
print(f"{len(future)} later trading days held out to score the forecast (never used for training)")

raw = pd.read_csv(cache_path(TICKER, START, END), parse_dates=["date"])
print("\nSplit events in the raw data (a 4:1 split is a 75% one-day 'crash' in unadjusted close):")
display(adjustment_events(raw))
plot_adjusted_vs_raw(raw, TICKER)
plt.show()

## 2. Walk-forward evaluation against baselines

Each fold is ordered in time as `train | validation | test`. The LSTM trains on `train`, stops early on `validation`,
and is scored on `test`, which it never influenced. Then the boundaries slide forward and it retrains from scratch, so
the score spans several market regimes. Every model predicts the same days, in dollars.

* **Persistence**: tomorrow's close = today's close.
* **MA(5), MA(20)**: average of the last 5 / 20 closes.
* **ARIMA(1,1,1)**: on log prices, fit once per fold on data before the test block.
* **LSTM**: mean of several random seeds (a small ensemble); the per-seed spread is shown below.

`RMSE_vs_Persistence` below 1 means better than persistence. The interval is a block bootstrap: if it contains 1.0,
the data cannot tell the model from the baseline.

In [ ]:
result = run_walk_forward(history, MODEL_CFG, WF_CFG)

In [ ]:
summary = result.summary(history)
n_days = summary.attrs["n_days"]
print(f"{n_days} out-of-sample days across {WF_CFG.n_folds} folds; {summary.attrs['up_day_rate_%']:.1f}% of them closed up")
print("(directional accuracy has to beat that up-day rate to mean anything; persistence makes no directional call)\n")
display(summary.round(3))

print("RMSE (USD) by fold, to see whether any edge is stable over time:")
display(result.per_fold_rmse(history).round(3))

print("Dollar RMSE of each individual LSTM seed (spread = run-to-run noise):")
display(result.seed_rmse(history).round(3).to_frame("RMSE").T)

In [ ]:
plot_walk_forward(history, result.predictions)
plt.show()
plot_ratio_ci(summary)
plt.show()

In [ ]:
row = summary.loc["LSTM"]
ratio, lo, hi = row["RMSE_vs_Persistence"], row["ratio_CI95_lo"], row["ratio_CI95_hi"]
verdict = ("better than" if hi < 1 else "worse than" if lo > 1 else "statistically indistinguishable from")
print(f"LSTM RMSE is {ratio:.3f}x persistence (95% CI {lo:.3f} to {hi:.3f}): {verdict} 'tomorrow = today' "
      f"over these {n_days} days.")
print(f"Lowest RMSE of all models: {summary['RMSE'].idxmin()}")
print(f"LSTM directional accuracy {row['DirAcc_%']:.1f}% vs {summary.attrs['up_day_rate_%']:.1f}% for always guessing 'up'")

## 3. Final models

For the forecast, train the same network on *all* history up to the origin (the last `val_size` days serve as the
early-stopping slice), once per seed.

In [ ]:
returns = log_returns(history).to_numpy()
final_models = [
    fit_model(returns, train_end=len(returns) - WF_CFG.val_size, val_end=len(returns), cfg=MODEL_CFG, seed=s)
    for s in range(WF_CFG.n_seeds)
]
print("epochs run per seed:", [m.epochs_run for m in final_models])

## 4. A 30-day forecast, as a fan of paths

Feeding a model's own prediction back in as input compounds error, so a single line overstates what is known. Instead
each simulated path adds a resampled **out-of-sample residual** (from the walk-forward test days above) to every
predicted return before feeding it back. The result is a median path with 80% and 95% bands, in dollars. The forecast
always starts from exactly the last `window` returns; nothing is padded or hard-coded.

In [ ]:
residuals = residuals_from_predictions(history, result.predictions["LSTM"])
fc = make_forecast(final_models, returns, origin_price=history.iloc[-1], origin_date=origin,
                   residuals=residuals, horizon=HORIZON, n_paths=2000)

lo80, hi80 = fc.band(0.80)
fan = pd.DataFrame({"forecast": fc.point, "median": fc.median, "lo80": lo80, "hi80": hi80},
                   index=pd.RangeIndex(1, HORIZON + 1, name="trading day"))
print(f"Last close (adjusted) on {origin:%Y-%m-%d}: {fc.origin_price:,.2f} USD")
display(fan.iloc[sorted({0, HORIZON // 3 - 1, 2 * HORIZON // 3 - 1, HORIZON - 1})].round(2))
plot_forecast(history, fc, ticker=TICKER)
plt.show()

## 5. Backtest: what actually happened

Trading days after the origin are matched by position (first close after the origin = forecast day 1). The baseline
is "flat at the last close". This is **one** 30-day path, so it illustrates the forecast; it is not a statistic. The
useful checks are whether the 80% / 95% bands cover roughly 80% / 95% of the realised closes, and whether the point
forecast beats doing nothing.

In [ ]:
if len(future) == 0:
    print(f"No closes after {origin:%Y-%m-%d} in the data yet; move END later to score the forecast.")
else:
    scored, fsum = score_forecast(fc, future)
    display(pd.Series(fsum).round(3).to_frame("value"))
    plot_forecast(history, fc, scored, TICKER)
    plt.show()

    better = "beat" if fsum["RMSE_ratio_vs_flat"] < 1 else "did not beat"
    print(f"Over {fsum['days_scored']} days the forecast {better} 'flat at the last close' "
          f"(RMSE ratio {fsum['RMSE_ratio_vs_flat']:.3f}).")
    print(f"Bands covered {fsum['coverage_80_%']:.0f}% (80% band) and {fsum['coverage_95_%']:.0f}% (95% band) "
          "of realised closes.")
    print("\nCaution: this is ONE 30-day path. Winning or losing against 'flat' on a single path is mostly luck, "
          "and consecutive days are strongly correlated, so band coverage here is not a calibration test either. "
          "Section 2 (many days, several folds, confidence intervals) is the evidence; this section is an illustration.")

## Limitations

* One asset, one market regime, one historical path; nothing here says how the model would do elsewhere or later.
* Adjusted history is rewritten by the data provider whenever a dividend or split is paid, so re-downloading later can
  change old values slightly. Keep the cached CSV if you need exact reproducibility.
* Predicting *direction* and magnitude of tomorrow's move from past prices alone is a very low signal-to-noise problem.
  If the intervals above include 1.0, that is the expected outcome, not a bug.
* No transaction costs, slippage or position sizing: this is a forecasting exercise, not a trading strategy.
